# Q3 · IA e Causa Raiz das Perdas (Text Analytics)
Pipeline de NLP para descobrir por que casos são julgados procedentes:
1. Embeddings dos campos `CDU_NAME` + `CX_PR_NAME` via OpenAI
2. Redução dimensional com t-SNE
3. Clustering com KMeans (elbow automático)
4. Labeling de cada cluster via GPT-4o-mini
5. Matriz Produto × Causa Raiz ordenada por impacto

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

load_dotenv()
client = OpenAI(api_key=os.getenv('API_OPENAI_KEY'))

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('OpenAI client inicializado.')

## 1 · Carregar base e segmentar por status

In [ ]:
aba2 = pd.read_csv('../outputs/aba2_limpa.csv')
aba2['RDR_EXTERNAL_CREATE_DTTM'] = pd.to_datetime(aba2['RDR_EXTERNAL_CREATE_DTTM'], format='mixed')
aba2['RDR_OUTGOING_DTTM']        = pd.to_datetime(aba2['RDR_OUTGOING_DTTM'], format='mixed', errors='coerce')

procedentes   = aba2[aba2['RDR_STATUS_NAME'].str.lower().str.contains('procedente') &
                     ~aba2['RDR_STATUS_NAME'].str.lower().str.contains('improcedente')].copy()
improcedentes = aba2[aba2['RDR_STATUS_NAME'].str.lower().str.contains('improcedente')].copy()
pendentes     = aba2[aba2['RDR_STATUS_NAME'].str.lower().str.contains('pendente')].copy()

print(f'Total RDR:       {len(aba2):,}')
print(f'  Procedentes:   {len(procedentes):,}  ({len(procedentes)/len(aba2)*100:.1f}%)')
print(f'  Improcedentes: {len(improcedentes):,}  ({len(improcedentes)/len(aba2)*100:.1f}%)')
print(f'  Pendentes:     {len(pendentes):,}  ({len(pendentes)/len(aba2)*100:.1f}%)')
print()
print(f'CDU_NAME:   {aba2.CDU_NAME.nunique()} valores únicos')
print(f'CX_PR_NAME: {aba2.CX_PR_NAME.nunique()} valores únicos')

## 2 · Comparação de textos: procedentes vs improcedentes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for ax, df_g, titulo, cor in [
    (axes[0, 0], procedentes,   'CDU_NAME — Procedentes',    '#d62728'),
    (axes[0, 1], improcedentes, 'CDU_NAME — Improcedentes',  '#2ca02c'),
    (axes[1, 0], procedentes,   'CX_PR_NAME — Procedentes',  '#d62728'),
    (axes[1, 1], improcedentes, 'CX_PR_NAME — Improcedentes','#2ca02c'),
]:
    col = 'CDU_NAME' if 'CDU' in titulo else 'CX_PR_NAME'
    top = df_g[col].value_counts().head(12)
    ax.barh(top.index[::-1], top.values[::-1], color=cor)
    ax.bar_label(ax.containers[0], padding=4, fontsize=8)
    ax.set_title(titulo, fontweight='bold', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Comparação: Procedentes vs Improcedentes', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/03_textos_comparacao.png', bbox_inches='tight')
plt.show()

## 3 · Taxa de procedência por CDU_NAME

In [ ]:
taxa_df = aba2[aba2['RDR_STATUS_NAME'].str.lower().str.contains('procedente|improcedente')].copy()
taxa_df['é_procedente'] = (~taxa_df['RDR_STATUS_NAME'].str.lower().str.contains('improcedente')).astype(int)

taxa_cdu = (
    taxa_df.groupby('CDU_NAME')
    .agg(total=('é_procedente', 'count'), procedentes=('é_procedente', 'sum'))
    .assign(taxa_proc=lambda x: x['procedentes'] / x['total'])
    .query('total >= 5')
    .sort_values('taxa_proc', ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 6))
top15 = taxa_cdu.head(15).reset_index()
cores = ['#d62728' if t > 0.5 else '#ff7f0e' if t > 0.25 else '#2ca02c'
         for t in top15['taxa_proc'][::-1]]
bars = ax.barh(top15['CDU_NAME'][::-1], top15['taxa_proc'][::-1], color=cores)
ax.bar_label(bars, labels=[f'{v:.0%}' for v in top15['taxa_proc'][::-1]], padding=4, fontsize=9)
ax.set_xlabel('Taxa de Procedência')
ax.set_title('Taxa de Procedência por CDU_NAME (mínimo 5 casos)', fontweight='bold')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='50%')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/03_taxa_procedencia_cdu.png', bbox_inches='tight')
plt.show()

print(taxa_cdu.head(15)[['total', 'procedentes', 'taxa_proc']]
      .style.format({'taxa_proc': '{:.0%}'}).to_string())

## 4 · Construir documento textual por caso

In [ ]:
procedentes['texto_doc'] = (
    'Causa: '    + procedentes['CDU_NAME'].fillna('Não informado').astype(str) +
    '. Processo: ' + procedentes['CX_PR_NAME'].fillna('Não informado').astype(str) +
    '. Tipo de cliente: ' + procedentes['RDR_USER_DOC_TYPE'].fillna('F')
      .map({'F': 'Pessoa Física', 'J': 'Pessoa Jurídica'}).fillna('Pessoa Física')
)

print(f'Total de documentos: {len(procedentes)}')
print(f'Textos únicos: {procedentes.texto_doc.nunique()}')
print()
for txt in procedentes['texto_doc'].sample(4, random_state=42):
    print(f'  → {txt}')

## 5 · Gerar embeddings via OpenAI

In [ ]:
EMBED_MODEL = 'text-embedding-3-small'

# Carregar cache se existir (evita custo de API em re-runs)
EMBED_PATH = '../outputs/03_embeddings.npy'
if os.path.exists(EMBED_PATH):
    embeddings = np.load(EMBED_PATH)
    print(f'Embeddings carregados do cache: {embeddings.shape}')
else:
    textos_unicos = procedentes['texto_doc'].unique().tolist()
    print(f'Gerando embeddings para {len(textos_unicos)} textos únicos...')

    def get_embeddings_batch(texts, batch_size=100):
        all_emb = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
            all_emb.extend([e.embedding for e in resp.data])
            print(f'  Batch {i//batch_size + 1}: {len(batch)} textos')
        return np.array(all_emb)

    emb_unicos = get_embeddings_batch(textos_unicos)
    mapa_embed = dict(zip(textos_unicos, emb_unicos))
    embeddings = np.array([mapa_embed[t] for t in procedentes['texto_doc']])

    np.save(EMBED_PATH, embeddings)
    print(f'Embeddings salvos. Shape: {embeddings.shape}')

## 6 · Redução dimensional com t-SNE

In [ ]:
print('Reduzindo com t-SNE...')
reducer = TSNE(
    n_components=2,
    perplexity=min(30, len(embeddings) - 1),
    learning_rate='auto',
    init='pca',
    random_state=42,
    max_iter=1000,
)
embedding_2d = reducer.fit_transform(embeddings)
print(f't-SNE concluído. Shape: {embedding_2d.shape}')

## 7 · Clustering com KMeans (elbow automático)

In [ ]:
K_range = range(3, min(12, len(embeddings) // 10 + 3))
inertias = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(embeddings)
    inertias.append(km.inertia_)

diffs    = np.diff(inertias)
rel_diff = diffs / inertias[:-1]
best_k   = list(K_range)[np.argmin(rel_diff) + 1]
best_k   = max(4, min(best_k, 8))
print(f'K ótimo (cotovelo): {best_k}')

clusterer = KMeans(n_clusters=best_k, random_state=42, n_init=20)
labels    = clusterer.fit_predict(embeddings)

procedentes = procedentes.copy()
procedentes['cluster'] = labels
n_clusters = best_k

print('\nTamanho por cluster:')
for c in sorted(set(labels)):
    grupo = procedentes[procedentes['cluster'] == c]
    top   = grupo['CDU_NAME'].value_counts().head(2).index.tolist()
    print(f'  Cluster {c} ({len(grupo)} casos): {top}')

## 8 · Visualização t-SNE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

cmap = plt.cm.get_cmap('tab10', n_clusters)
for lbl in sorted(set(labels)):
    mask = labels == lbl
    axes[0].scatter(embedding_2d[mask, 0], embedding_2d[mask, 1],
                    c=[cmap(lbl)], s=40, alpha=0.7, label=f'Cluster {lbl}')
axes[0].set_title('t-SNE — Clusters KMeans', fontweight='bold')
axes[0].set_xlabel('Dim 1'); axes[0].set_ylabel('Dim 2')
axes[0].legend(fontsize=9)

top_causas = procedentes['CDU_NAME'].value_counts().head(8).index.tolist()
cmap2 = plt.cm.get_cmap('Set1', len(top_causas))
outros = ~procedentes['CDU_NAME'].isin(top_causas)
axes[1].scatter(embedding_2d[outros, 0], embedding_2d[outros, 1],
                c='lightgray', s=25, alpha=0.4, label='Outras causas')
for i, causa in enumerate(top_causas):
    mask_c = (procedentes['CDU_NAME'] == causa).values
    axes[1].scatter(embedding_2d[mask_c, 0], embedding_2d[mask_c, 1],
                    c=[cmap2(i)], s=50, alpha=0.8,
                    label=causa[:30] + ('...' if len(causa) > 30 else ''))
axes[1].set_title('t-SNE — CDU_NAME (top 8)', fontweight='bold')
axes[1].set_xlabel('Dim 1'); axes[1].set_ylabel('Dim 2')
axes[1].legend(fontsize=7)

plt.suptitle('Espaço Semântico dos Procedentes Bacen', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/03_tsne_clusters.png', bbox_inches='tight')
plt.show()

## 9 · Labeling dos clusters via GPT-4o-mini

In [ ]:
def label_cluster(cluster_id, df_cluster):
    causas    = df_cluster['CDU_NAME'].value_counts().head(10)
    processos = df_cluster['CX_PR_NAME'].value_counts().head(5)
    pf_pct    = df_cluster['RDR_USER_DOC_TYPE'].value_counts(normalize=True).get('F', 0) * 100

    prompt = f"""Você é especialista em regulação financeira e atendimento do Mercado Pago.

Analise um grupo de {len(df_cluster)} reclamações julgadas PROCEDENTES pelo Bacen.
Perfil: {pf_pct:.0f}% Pessoa Física.

Principais CAUSAS (CDU_NAME):
{chr(10).join(f'- {n} ({v} casos)' for n, v in causas.items())}

Principais PROCESSOS (CX_PR_NAME):
{chr(10).join(f'- {n} ({v} casos)' for n, v in processos.items())}

Responda APENAS em JSON com as chaves:
- "label": nome curto acionável (máx 5 palavras)
- "causa_raiz": por que o Bacen julga procedente (2-3 frases)
- "produto_mp": produto mais afetado (Pix, Cartão de Crédito, Empréstimo, Conta, Múltiplos)
- "acao_recomendada": ação concreta para reduzir as perdas (2-3 frases)
- "urgencia": Alta, Média ou Baixa"""

    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.1,
        response_format={'type': 'json_object'},
    )
    return json.loads(resp.choices[0].message.content)


cluster_labels = {}
print(f'Labeling de {n_clusters} clusters via GPT-4o-mini...\n')
for cid in sorted(set(labels)):
    df_c = procedentes[procedentes['cluster'] == cid]
    print(f'Cluster {cid} ({len(df_c)} casos):', end=' ')
    resultado = label_cluster(cid, df_c)
    cluster_labels[cid] = resultado
    print(f'{resultado.get("label")} | {resultado.get("produto_mp")} | {resultado.get("urgencia")}')

## 10 · Análise do menor cluster

In [ ]:
contagem     = procedentes['cluster'].value_counts()
menor_c      = contagem.idxmin()
grupo_menor  = procedentes[procedentes['cluster'] == menor_c]

print(f'Menor cluster: {menor_c} ({len(grupo_menor)} casos)')
print()
print('CDU_NAME:')
print(grupo_menor['CDU_NAME'].value_counts().head(8).to_string())
print()
print('CX_PR_NAME:')
print(grupo_menor['CX_PR_NAME'].value_counts().head(5).to_string())

## 11 · Matriz Produto × Causa Raiz

In [ ]:
total_proc = len(procedentes)
rows = []
for cid, info in cluster_labels.items():
    df_c = procedentes[procedentes['cluster'] == cid]
    top_causas = ', '.join(df_c['CDU_NAME'].value_counts().head(3).index.tolist())
    rows.append({
        'cluster':             cid,
        'label_llm':           info.get('label', f'Cluster {cid}'),
        'produto':             info.get('produto_mp', 'Outro'),
        'urgencia':            info.get('urgencia', 'Média'),
        'causa_raiz_llm':      info.get('causa_raiz', ''),
        'acao_recomendada':    info.get('acao_recomendada', ''),
        'top_cdu_names':       top_causas,
        'volume_procedentes':  len(df_c),
        'pct_do_indice':       round(len(df_c) / total_proc * 100, 1),
    })

df_result = pd.DataFrame(rows).sort_values('volume_procedentes', ascending=False)

print('=== Clusters de Causa Raiz ===')
print(df_result[['produto', 'label_llm', 'urgencia', 'volume_procedentes', 'pct_do_indice', 'top_cdu_names']]
      .to_string(index=False))

df_result.to_csv('../outputs/03_cluster_analise.csv', index=False)
print('\nExportado para ../outputs/03_cluster_analise.csv')

## 12 · Heatmap e gráfico de barras

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, max(5, len(df_result))))

cores_urg = {'Alta': '#d62728', 'Média': '#ff7f0e', 'Baixa': '#2ca02c'}
df_sorted = df_result.sort_values('volume_procedentes')
cores = [cores_urg.get(u, '#1f77b4') for u in df_sorted['urgencia']]

axes[0].barh(df_sorted['label_llm'], df_sorted['volume_procedentes'], color=cores)
axes[0].bar_label(axes[0].containers[0],
                  labels=[f'{v} ({p}%)' for v, p in
                          zip(df_sorted['volume_procedentes'], df_sorted['pct_do_indice'])],
                  padding=4, fontsize=9)
axes[0].set_title('Volume de Procedentes por Grupo\n(vermelho = urgência alta)', fontweight='bold')
axes[0].tick_params(axis='y', labelsize=9)

if df_result['produto'].nunique() > 1:
    pivot = df_result.pivot_table(index='produto', columns='label_llm',
                                  values='volume_procedentes', aggfunc='sum', fill_value=0)
    sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1],
                linewidths=0.5, cbar_kws={'label': 'Procedentes'})
    axes[1].set_title('Matriz Produto × Causa Raiz', fontweight='bold')
    axes[1].tick_params(axis='x', labelsize=8, rotation=30)
else:
    axes[1].bar(range(len(df_result)), df_result['volume_procedentes'],
                color='#d62728', alpha=0.7)
    axes[1].set_xticks(range(len(df_result)))
    axes[1].set_xticklabels(df_result['label_llm'], rotation=35, ha='right', fontsize=8)
    axes[1].set_title('Volume por Causa Raiz', fontweight='bold')

plt.suptitle('Causa Raiz dos Procedentes — Análise por IA', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/03_heatmap_produto_causa.png', bbox_inches='tight')
plt.show()

## 13 · Resumo executivo

In [ ]:
print('=' * 70)
print('RESUMO EXECUTIVO — CAUSA RAIZ DOS PROCEDENTES')
print('=' * 70)

for _, row in df_result[df_result['urgencia'] == 'Alta'] \
        .sort_values('volume_procedentes', ascending=False).iterrows():
    print(f'\n[{row["produto"]}] {row["label_llm"]} — {row["volume_procedentes"]} casos ({row["pct_do_indice"]}%)')
    print(f'  Causa raiz: {row["causa_raiz_llm"]}')
    print(f'  Ação:       {row["acao_recomendada"]}')

print(f'\nTotal analisado: {total_proc} procedentes')